## Basic Imports

In [ ]:
!pip install scorch

In [ ]:

from scorch import scores
from statistics import mean
import os
import json
import random
import numpy as np
import random
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# GPU setup
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



Using device: cuda


## DATA LOADING AND PREPROCESSING

In [ ]:
def load_data(data_path='train_data.jsonl',
              labels_path='train_labels.json'):
    """Load training data and labels"""
    print("Loading data...")

    # Load mentions
    mentions = {}
    with open(data_path, 'r') as f:
        for line in f:
            mention_data = json.loads(line.strip())
            mentions[mention_data['mention_id']] = mention_data

    # Load training clusters
    with open(labels_path, 'r') as f:
        clusters = json.load(f)

    # Create mappings
    mention_to_cluster = {}
    cluster_to_mentions = defaultdict(list)
    for cluster_id, cluster_mentions in enumerate(clusters):
        for mid in cluster_mentions:
            mention_to_cluster[mid] = cluster_id
            cluster_to_mentions[cluster_id].append(mid)

    print(f"Loaded {len(mentions)} mentions and {len(clusters)} clusters.")
    return mentions, clusters, mention_to_cluster, cluster_to_mentions


def preprocess_mention(mention_data):
    """
    Format mention with context for SciBERT:
    [MENTION] text [TYPE] type [REL] relations [SEP] sentence
    """
    mention_text = mention_data['mention']
    mention_type = mention_data['type']
    relations = mention_data['relations']
    sentence = mention_data['sentence']

    rel_str = ""
    if relations:
        rel_parts = [f"{rel['mention']}" for rel in relations]
        rel_str = " | ".join(rel_parts)

    input_str = f"[MENTION] {mention_text} [TYPE] {mention_type} [REL] {rel_str} [SEP] {sentence}"
    return input_str


def cluster_based_split(clusters, test_size=0.2, val_size=0.5, random_state=42):
    """
    Split data by clusters (not mentions) to prevent data leakage.

    Args:
        clusters: List of clusters (each cluster is a list of mention IDs)
        test_size: Proportion for temp split (val + test)
        val_size: Proportion of temp for validation
        random_state: Random seed

    Returns:
        train_cluster_ids, val_cluster_ids, test_cluster_ids
    """
    cluster_ids = list(range(len(clusters)))

    # 80% train, 20% temp (val + test)
    train_cluster_ids, temp_cluster_ids = train_test_split(
        cluster_ids, test_size=test_size, random_state=random_state
    )

    # Split temp into 50% val, 50% test (each is 10% of total)
    val_cluster_ids, test_cluster_ids = train_test_split(
        temp_cluster_ids, test_size=val_size, random_state=random_state
    )

    return train_cluster_ids, val_cluster_ids, test_cluster_ids


## DATASET WITH EFFICIENT COLLATION


In [ ]:

class CorefPairDataset(Dataset):
    """
    Dataset for contrastive learning of coreference resolution.
    Returns raw text data; tokenization happens in collate_fn for efficiency.
    """
    def __init__(self, mention_ids, mention_inputs, mention_to_cluster,
                 num_negatives=2):
        self.mention_ids = mention_ids
        self.mention_inputs = mention_inputs
        self.mention_to_cluster = mention_to_cluster
        self.num_negatives = num_negatives

    def __len__(self):
        return len(self.mention_ids)

    def __getitem__(self, idx):
        anchor_id = self.mention_ids[idx]
        anchor_cluster = self.mention_to_cluster[anchor_id]

        # Positive: random from same cluster (or self if singleton)
        same_cluster = [mid for mid in self.mention_ids
                       if self.mention_to_cluster[mid] == anchor_cluster
                       and mid != anchor_id]
        pos_id = random.choice(same_cluster) if same_cluster else anchor_id

        # Negatives: random from different clusters
        diff_clusters = [mid for mid in self.mention_ids
                        if self.mention_to_cluster[mid] != anchor_cluster]
        neg_ids = random.sample(diff_clusters, min(self.num_negatives, len(diff_clusters)))

        return {
            'anchor_text': self.mention_inputs[anchor_id],
            'positive_text': self.mention_inputs[pos_id],
            'negative_texts': [self.mention_inputs[nid] for nid in neg_ids]
        }


def collate_fn(batch, tokenizer, max_length=256):
    """
    Efficient batch collation with batch tokenization.
    This is 3-5x faster than tokenizing in __getitem__.
    """
    # Extract texts
    anchors = [item['anchor_text'] for item in batch]
    positives = [item['positive_text'] for item in batch]

    # Flatten negatives (batch_size * num_negatives texts)
    all_negatives = []
    for item in batch:
        all_negatives.extend(item['negative_texts'])

    num_negatives = len(batch[0]['negative_texts'])

    # Batch tokenize (much faster!)
    anchor_enc = tokenizer(
        anchors, padding=True, truncation=True,
        max_length=max_length, return_tensors='pt'
    )

    positive_enc = tokenizer(
        positives, padding=True, truncation=True,
        max_length=max_length, return_tensors='pt'
    )

    negative_enc = tokenizer(
        all_negatives, padding=True, truncation=True,
        max_length=max_length, return_tensors='pt'
    )

    return {
        'anchor': anchor_enc,
        'positive': positive_enc,
        'negatives': negative_enc,
        'num_negatives': num_negatives
    }


## MODEL ARCHITECTURE

In [ ]:
class SiameseSciBERT(nn.Module):
    """
    Siamese network with SciBERT encoder and improved projection head.

    Improvements:
    - Added BatchNorm for training stability
    - Added Dropout for regularization
    - L2 normalization in forward pass
    """
    def __init__(self, embedding_dim=768, projection_dim=256, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')

        # Improved projection head
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim)
        )

    def forward(self, input_ids, attention_mask):
        # Get CLS embedding from SciBERT
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]

        # Project and normalize
        proj_emb = self.projection(cls_emb)
        return F.normalize(proj_emb, p=2, dim=1)  # L2 normalize

## LOSS FUNCTION


In [ ]:
class InfoNCELoss(nn.Module):
    """
    InfoNCE (NT-Xent) loss using PyTorch built-in functions.

    More efficient and numerically stable than custom implementation.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, anchor, positive, negatives, num_negatives):
        """
        Args:
            anchor: [batch_size, dim]
            positive: [batch_size, dim]
            negatives: [batch_size * num_negatives, dim]
            num_negatives: int, number of negatives per anchor
        """
        batch_size = anchor.size(0)

        # Embeddings are already normalized in model forward
        # Positive similarity: [batch_size, 1]
        pos_sim = torch.sum(anchor * positive, dim=1, keepdim=True) / self.temperature

        # Negative similarity: [batch_size, num_negatives]
        negatives_reshaped = negatives.view(batch_size, num_negatives, -1)
        neg_sim = torch.bmm(
            anchor.unsqueeze(1),  # [batch_size, 1, dim]
            negatives_reshaped.transpose(1, 2)  # [batch_size, dim, num_negatives]
        ).squeeze(1) / self.temperature  # [batch_size, num_negatives]

        # Logits: [batch_size, 1 + num_negatives]
        logits = torch.cat([pos_sim, neg_sim], dim=1)

        # Labels: positive is always at index 0
        labels = torch.zeros(batch_size, dtype=torch.long, device=anchor.device)

        return self.cross_entropy(logits, labels)

## TRAINING AND EVALUATION

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    """Train for one epoch"""
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        # Move to device
        anchor = {k: v.to(device) for k, v in batch['anchor'].items()}
        positive = {k: v.to(device) for k, v in batch['positive'].items()}
        negatives = {k: v.to(device) for k, v in batch['negatives'].items()}
        num_negatives = batch['num_negatives']

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        # Mixed precision forward pass
        with torch.amp.autocast('cuda'):
            anchor_emb = model(anchor['input_ids'], anchor['attention_mask'])
            pos_emb = model(positive['input_ids'], positive['attention_mask'])
            neg_embs = model(negatives['input_ids'], negatives['attention_mask'])

            loss = criterion(anchor_emb, pos_emb, neg_embs, num_negatives)

        # Backward with gradient scaling and clipping
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def get_embeddings(model, mention_ids, mention_inputs, tokenizer, batch_size=16):
    """Generate embeddings for mentions"""
    model.eval()
    embeddings = {}
    dataloader = DataLoader(mention_ids, batch_size=batch_size)

    with torch.no_grad():
        for batch_ids in tqdm(dataloader, desc="Getting embeddings"):
            batch_texts = [mention_inputs[mid] for mid in batch_ids]
            encodings = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=512, return_tensors='pt'
            ).to(device)

            with torch.amp.autocast('cuda'):
                embs = model(encodings['input_ids'], encodings['attention_mask'])

            for mid, emb in zip(batch_ids, embs):
                embeddings[mid] = emb.cpu().numpy()

    return embeddings


def cluster_mentions(embeddings, threshold=0.8):
    """Cluster mentions using graph-based connected components"""
    mention_list = list(embeddings.keys())
    sim_matrix = cosine_similarity(np.array(list(embeddings.values())))

    # Build graph
    G = nx.Graph()
    G.add_nodes_from(mention_list)
    for i in range(len(mention_list)):
        for j in range(i+1, len(mention_list)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(mention_list[i], mention_list[j])

    # Get connected components
    predicted_clusters = [list(component) for component in nx.connected_components(G)]
    return predicted_clusters


def find_optimal_threshold(model, val_ids, mention_inputs, gold_clusters,
                           tokenizer, metric_fn):
    """Find best clustering threshold on validation set"""
    embeddings = get_embeddings(model, val_ids, mention_inputs, tokenizer)

    best_threshold = 0.5
    best_f1 = 0

    print("Tuning threshold on validation set...")
    for threshold in np.arange(0.3, 0.95, 0.05):
        predicted = cluster_mentions(embeddings, threshold=threshold)
        f1 = metric_fn(gold_clusters, predicted)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    print(f"Optimal threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")
    return best_threshold


def evaluate(model, val_ids, mention_inputs, gold_clusters, tokenizer,
             threshold, metric_fn):
    """Evaluate model on validation/test set"""
    embeddings = get_embeddings(model, val_ids, mention_inputs, tokenizer)
    predicted_clusters = cluster_mentions(embeddings, threshold=threshold)
    f1 = metric_fn(gold_clusters, predicted_clusters)
    return f1


## METRICS


In [ ]:
try:
    from scorch import scores as scorch_scores
    from statistics import mean as _mean
    _SCORCH_AVAILABLE = True
except ImportError:
    _SCORCH_AVAILABLE = False


def evaluate_with_scorch(gold_clusters, pred_clusters):
    """
    Compute CoNLL F1 using the scorch library.
    Returns a single float (CoNLL F1 = mean of MUC, B³, CEAFe F1 scores).

    Falls back to a B³ approximation if scorch is not installed.
    Install with: pip install scorch
    """
    if _SCORCH_AVAILABLE:
        # scorch expects sequences of sets
        gold = [set(c) for c in gold_clusters]
        pred = [set(c) for c in pred_clusters]

        _, _, muc_f  = scorch_scores.muc(gold, pred)
        _, _, b3_f   = scorch_scores.b_cubed(gold, pred)
        _, _, ceaf_f = scorch_scores.ceaf_e(gold, pred)

        return _mean([muc_f, b3_f, ceaf_f])
    else:
        # Fallback: simplified B³ F1
        def get_partition(clusters):
            m_to_c = {}
            for cid, cl in enumerate(clusters):
                for mid in cl:
                    m_to_c[mid] = cid
            return m_to_c

        m_to_c_gold = get_partition(gold_clusters)
        m_to_c_pred = get_partition(pred_clusters)
        all_mentions = set(m_to_c_gold.keys())

        if not all_mentions:
            return 0.0

        precision_sum = recall_sum = 0
        for m in all_mentions:
            gold_cl = set(mm for mm in all_mentions
                          if m_to_c_gold.get(mm) == m_to_c_gold.get(m))
            pred_cl = set(mm for mm in all_mentions
                          if m_to_c_pred.get(mm) == m_to_c_pred.get(m))
            intersection = len(gold_cl & pred_cl)
            precision_sum += intersection / len(pred_cl) if pred_cl else 0
            recall_sum    += intersection / len(gold_cl) if gold_cl else 0

        p = precision_sum / len(all_mentions)
        r = recall_sum    / len(all_mentions)
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

## MAIN TRAINING SCRIPT


In [ ]:
def main():
    print("=" * 80)
    print("Baseline SciBERT Coreference Resolution Training")
    print("=" * 80)

    # Load data
    mentions, clusters, mention_to_cluster, cluster_to_mentions = load_data()

    # Preprocess mentions
    print("\nPreprocessing mentions...")
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}

    # Cluster-based split (prevents data leakage)
    print("\nSplitting data by clusters (train/val/test = 80/10/10)...")
    train_cluster_ids, val_cluster_ids, test_cluster_ids = cluster_based_split(clusters)

    # Get mention IDs for each split
    train_ids = [mid for cid in train_cluster_ids for mid in clusters[cid]]
    val_ids = [mid for cid in val_cluster_ids for mid in clusters[cid]]
    test_ids = [mid for cid in test_cluster_ids for mid in clusters[cid]]

    print(f"Train: {len(train_ids)} mentions from {len(train_cluster_ids)} clusters")
    print(f"Val:   {len(val_ids)} mentions from {len(val_cluster_ids)} clusters")
    print(f"Test:  {len(test_ids)} mentions from {len(test_cluster_ids)} clusters")

    # Get gold clusters for val and test
    val_gold_clusters = [clusters[cid] for cid in val_cluster_ids]
    test_gold_clusters = [clusters[cid] for cid in test_cluster_ids]

    # Initialize tokenizer
    print("\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Create datasets with collate function
    print("\nCreating datasets...")
    train_dataset = CorefPairDataset(train_ids, mention_inputs, mention_to_cluster)

    # Custom collate function with tokenizer
    def collate_with_tokenizer(batch):
        return collate_fn(batch, tokenizer)

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        collate_fn=collate_with_tokenizer,
        num_workers=2,
        pin_memory=True
    )

    # Initialize model
    print("\nInitializing model...")
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.amp.GradScaler('cuda')
    criterion = InfoNCELoss(temperature=0.07)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Training loop
    print("\n" + "=" * 80)
    print("Training")
    print("=" * 80)

    num_epochs = 50
    patience = 3
    best_val_f1 = 0
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")

        # Train
        train_loss = train_epoch(model, train_loader, optimizer, scheduler,
                                scaler, criterion)

        # Evaluate on validation set every epoch
        if epoch % 1 == 0:  # Can change frequency
            # Tune threshold on first evaluation, then use fixed
            if epoch == 1:
                best_threshold = find_optimal_threshold(
                    model, val_ids, mention_inputs, val_gold_clusters,
                    tokenizer, evaluate_with_scorch
                )

            val_f1 = evaluate(
                model, val_ids, mention_inputs, val_gold_clusters,
                tokenizer, best_threshold, evaluate_with_scorch
            )

            print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val F1 = {val_f1:.4f}")

            # Early stopping
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), 'best_model_refactored.pth')
                print("✓ Best model saved.")
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

    # Final evaluation on test set
    print("\n" + "=" * 80)
    print("Final Evaluation on Test Set")
    print("=" * 80)

    model.load_state_dict(torch.load('best_model_refactored.pth'))

    # Tune threshold on validation set for final test
    best_threshold = find_optimal_threshold(
        model, val_ids, mention_inputs, val_gold_clusters,
        tokenizer, evaluate_with_scorch
    )

    test_f1 = evaluate(
        model, test_ids, mention_inputs, test_gold_clusters,
        tokenizer, best_threshold, evaluate_with_scorch
    )

    print(f"\nTest F1 (with threshold {best_threshold:.2f}): {test_f1:.4f}")
    print("\n" + "=" * 80)
    print("Training complete!")
    print("=" * 80)


if __name__ == "__main__":
    main()

Baseline SciBERT Coreference Resolution Training
Loading data...
Loaded 2974 mentions and 733 clusters.

Preprocessing mentions...

Splitting data by clusters (train/val/test = 80/10/10)...
Train: 2219 mentions from 586 clusters
Val:   187 mentions from 73 clusters
Test:  568 mentions from 74 clusters

Loading tokenizer...

Creating datasets...

Initializing model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 72028.52it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model parameters: 110,182,144

Training

Epoch 1/50


Getting embeddings: 100%|██████████| 12/12 [00:00<00:00, 150.23it/s]


Tuning threshold on validation set...
Optimal threshold: 0.65, F1: 0.9734


...

Epoch 9: Train Loss = 0.0004, Val F1 = 0.9602

Early stopping at epoch 9

Final Evaluation on Test Set


Getting embeddings: 100%|██████████| 12/12 [00:00<00:00, 158.48it/s]


Tuning threshold on validation set...
Optimal threshold: 0.70, F1: 0.9816


Getting embeddings: 100%|██████████| 36/36 [00:00<00:00, 174.44it/s]


Test F1 (with threshold 0.70): 0.9968

Training complete!


## Predicting the test

In [ ]:
# from cross_encoder import CrossEncoderReranker, rerank_clusters

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CONFIGURATION - Edit these values to customize behavior
# ============================================================================
CONFIG = {
    'model_path': 'best_model_refactored.pth',           # Path to trained bi-encoder
    'test_data_path': 'test_data.jsonl', # Path to test data
    'threshold': 0.8,                                     # Clustering threshold (None = auto 0.8)
    'cross_encoder_path': None,                          # Path to cross-encoder (None = disabled)
    'ce_threshold': 0.5,                                 # Cross-encoder threshold
    'output_path': 'clusters.json',                      # Output clusters file
}
# ============================================================================


def load_test_data(test_data_path='test_data.jsonl'):
    """
    Load test data (without labels).

    Returns:
        mentions: Dict mapping mention_id -> mention data
        mention_inputs: Dict mapping mention_id -> preprocessed text
        mention_ids: List of all mention IDs
    """
    print(f"Loading test data from {test_data_path}...")

    mentions = {}
    with open(test_data_path, 'r') as f:
        for line in f:
            mention_data = json.loads(line.strip())
            mentions[mention_data['mention_id']] = mention_data

    # Preprocess mentions
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}
    mention_ids = list(mentions.keys())

    print(f"Loaded {len(mentions)} test mentions.")
    return mentions, mention_inputs, mention_ids


def predict_clusters(model_path, test_data_path='test_data.jsonl',
                     threshold=None, cross_encoder_path=None, ce_threshold=0.5):
    """
    Predict clusters for test data using trained model.

    Args:
        model_path: Path to trained bi-encoder model (.pth file)
        test_data_path: Path to test_data.jsonl
        threshold: Clustering threshold (if None, uses 0.8 as default)
        cross_encoder_path: Optional path to cross-encoder for re-ranking
        ce_threshold: Cross-encoder threshold (default 0.5)

    Returns:
        predicted_clusters: List of lists, each containing mention IDs in a cluster
    """
    # Load test data
    mentions, mention_inputs, mention_ids = load_test_data(test_data_path)

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Load bi-encoder model
    print(f"Loading bi-encoder model from {model_path}...")
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Get embeddings
    print("Generating embeddings...")
    embeddings = get_embeddings(model, mention_ids, mention_inputs, tokenizer, batch_size=16)

    # Cluster using bi-encoder
    if threshold is None:
        threshold = 0.8  # Default threshold
        print(f"Using default threshold: {threshold}")
    else:
        print(f"Using threshold: {threshold}")

    print("Clustering mentions...")
    predicted_clusters = cluster_mentions(embeddings, threshold=threshold)
    print(f"Found {len(predicted_clusters)} clusters")

    # Optional: Re-rank with cross-encoder
    if cross_encoder_path is not None:
        print(f"\nLoading cross-encoder from {cross_encoder_path}...")
        cross_encoder = CrossEncoderReranker().to(device)
        cross_encoder.load_state_dict(torch.load(cross_encoder_path, map_location=device))
        cross_encoder.eval()

        print(f"Re-ranking clusters with cross-encoder (threshold={ce_threshold})...")
        predicted_clusters = rerank_clusters(
            predicted_clusters, mention_inputs, cross_encoder,
            tokenizer, device, threshold=ce_threshold
        )
        print(f"After re-ranking: {len(predicted_clusters)} clusters")

    return predicted_clusters


def save_clusters(predicted_clusters, output_path='clusters.json'):
    """
    Save predicted clusters to JSON file.

    Format: List of lists, where each sublist contains mention IDs in a cluster.
    Example: [["m1", "m2", "m3"], ["m4", "m5"], ["m6"]]

    Args:
        predicted_clusters: List of lists of mention IDs
        output_path: Path to output JSON file
    """
    print(f"\nSaving clusters to {output_path}...")

    with open(output_path, 'w') as f:
        json.dump(predicted_clusters, f, indent=2)

    # Print statistics
    cluster_sizes = [len(cluster) for cluster in predicted_clusters]
    print(f"\nCluster Statistics:")
    print(f"  Total clusters: {len(predicted_clusters)}")
    print(f"  Singleton clusters: {sum(1 for size in cluster_sizes if size == 1)}")
    print(f"  Average cluster size: {sum(cluster_sizes) / len(cluster_sizes):.2f}")
    print(f"  Largest cluster: {max(cluster_sizes)}")
    print(f"\nClusters saved successfully!")


"""Run inference with configuration from CONFIG dictionary."""
print("=" * 80)
print("SciBERT Coreference Resolution - Test Inference")
print("=" * 80)
print(f"Model: {CONFIG['model_path']}")
print(f"Test data: {CONFIG['test_data_path']}")
print(f"Threshold: {CONFIG['threshold']}")
print(f"Output: {CONFIG['output_path']}")
if CONFIG['cross_encoder_path']:
    print(f"Cross-encoder: {CONFIG['cross_encoder_path']}")
    print(f"Cross-encoder threshold: {CONFIG['ce_threshold']}")
print("=" * 80)

# Predict clusters
predicted_clusters = predict_clusters(
    model_path=CONFIG['model_path'],
    test_data_path=CONFIG['test_data_path'],
    threshold=CONFIG['threshold'],
    cross_encoder_path=CONFIG['cross_encoder_path'],
    ce_threshold=CONFIG['ce_threshold']
)

# Save to JSON
save_clusters(predicted_clusters, output_path=CONFIG['output_path'])

print("\n" + "=" * 80)
print("Inference complete!")
print("=" * 80)




SciBERT Coreference Resolution - Test Inference
Model: best_model_refactored.pth
Test data: test_data.jsonl
Threshold: 0.8
Output: clusters.json
Loading test data from test_data.jsonl...
Loaded 743 test mentions.
Loading tokenizer...
Loading bi-encoder model from best_model_refactored.pth...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62084.68it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Getting embeddings: 100%|██████████| 47/47 [00:00<00:00, 165.53it/s]


Using threshold: 0.8
Clustering mentions...
Found 244 clusters

Saving clusters to clusters.json...

Cluster Statistics:
  Total clusters: 244
  Singleton clusters: 136
  Average cluster size: 3.05
  Largest cluster: 73

Clusters saved successfully!

Inference complete!


## Cross encoder

In [ ]:
"""
Cross-Encoder Re-Ranking for SciBERT Coreference Resolution

This module implements a cross-encoder that can be used to re-rank
bi-encoder clustering results for higher precision.
"""


class CrossEncoderReranker(nn.Module):
    """
    Cross-encoder for pairwise coreference classification.

    Takes concatenated mention pairs as input and predicts whether
    they are coreferent (binary classification).
    """
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)  # Binary score
        )

    def forward(self, input_ids, attention_mask):
        """
        Input: [CLS] mention1_text [SEP] mention2_text [SEP]
        Output: Coreference score (logit)
        """
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]
        score = self.classifier(cls_emb).squeeze(-1)
        return score


class PairwiseCorefDataset(Dataset):
    """
    Dataset for training cross-encoder with positive and negative pairs.
    """
    def __init__(self, mention_ids, mention_inputs, mention_to_cluster,
                 cluster_to_mentions):
        """
        Creates balanced positive and negative pairs.
        """
        self.mention_inputs = mention_inputs
        self.pairs = []
        self.labels = []

        # Positive pairs (same cluster)
        for cluster_id, cluster_mentions in cluster_to_mentions.items():
            if len(cluster_mentions) < 2:
                continue
            for i in range(len(cluster_mentions)):
                for j in range(i+1, len(cluster_mentions)):
                    self.pairs.append((cluster_mentions[i], cluster_mentions[j]))
                    self.labels.append(1)

        # Negative pairs (different clusters) - sample to balance
        num_positives = len(self.pairs)
        all_mentions = list(mention_ids)
        negatives_added = 0
        max_attempts = num_positives * 10  # Prevent infinite loop
        attempts = 0

        while negatives_added < num_positives and attempts < max_attempts:
            mid1, mid2 = random.sample(all_mentions, 2)
            if mention_to_cluster[mid1] != mention_to_cluster[mid2]:
                self.pairs.append((mid1, mid2))
                self.labels.append(0)
                negatives_added += 1
            attempts += 1

        print(f"Created pairwise dataset: {len(self.pairs)} pairs ({num_positives} pos, {negatives_added} neg)")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        mid1, mid2 = self.pairs[idx]
        return {
            'mention1': self.mention_inputs[mid1],
            'mention2': self.mention_inputs[mid2],
            'label': self.labels[idx]
        }


def cross_encoder_collate_fn(batch, tokenizer, max_length=512):
    """
    Collate function for cross-encoder training.
    Concatenates mention pairs: [CLS] m1 [SEP] m2 [SEP]
    """
    texts = [f"{item['mention1']} [SEP] {item['mention2']}" for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    }


def train_cross_encoder(train_dataset, val_dataset, tokenizer, device,
                       num_epochs=15, batch_size=8, lr=2e-5):
    """
    Train cross-encoder for pairwise coreference classification.

    Returns:
        Trained CrossEncoderReranker model
    """
    model = CrossEncoderReranker().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda')

    # Collate function
    def collate_fn(batch):
        return cross_encoder_collate_fn(batch, tokenizer)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True
    )

    best_val_acc = 0
    patience = 0
    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch in tqdm(train_loader, desc=f"Training CE Epoch {epoch}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            # Accuracy
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validating CE Epoch {epoch}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                with torch.amp.autocast('cuda'):
                    logits = model(input_ids, attention_mask)

                preds = (torch.sigmoid(logits) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += len(labels)

        val_acc = val_correct / val_total

        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_cross_encoder.pth')
            print("✓ Best cross-encoder saved.")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    return model


def rerank_clusters(bi_encoder_clusters, mention_inputs, cross_encoder,
                   tokenizer, device, threshold=0.5):
    """
    Re-rank cluster assignments using cross-encoder.

    Strategy: For each bi-encoder cluster, re-score all pairs and rebuild
    cluster using graph-based connected components.

    Args:
        bi_encoder_clusters: List of clusters from bi-encoder
        mention_inputs: Dict of mention texts
        cross_encoder: Trained CrossEncoderReranker
        tokenizer: SciBERT tokenizer
        device: cuda/cpu
        threshold: Score threshold for keeping a pair (default 0.5)

    Returns:
        refined_clusters: List of re-ranked clusters
    """
    cross_encoder.eval()
    refined_clusters = []

    for cluster in tqdm(bi_encoder_clusters, desc="Re-ranking clusters"):
        if len(cluster) <= 1:
            refined_clusters.append(cluster)
            continue

        # Build pairs to re-evaluate (all pairs in cluster)
        pairs_to_check = [(cluster[i], cluster[j])
                         for i in range(len(cluster))
                         for j in range(i+1, len(cluster))]

        # Score with cross-encoder
        scores = []
        batch_size = 16

        with torch.no_grad():
            for i in range(0, len(pairs_to_check), batch_size):
                batch_pairs = pairs_to_check[i:i+batch_size]

                # Prepare texts
                texts = [f"{mention_inputs[mid1]} [SEP] {mention_inputs[mid2]}"
                        for mid1, mid2 in batch_pairs]

                enc = tokenizer(
                    texts, padding=True, truncation=True,
                    max_length=512, return_tensors='pt'
                ).to(device)

                with torch.amp.autocast('cuda'):
                    logits = cross_encoder(enc['input_ids'], enc['attention_mask'])
                    batch_scores = torch.sigmoid(logits).cpu().numpy()

                for (mid1, mid2), score in zip(batch_pairs, batch_scores):
                    scores.append((mid1, mid2, score))

        # Rebuild cluster keeping only confident pairs
        G = nx.Graph()
        G.add_nodes_from(cluster)
        for mid1, mid2, score in scores:
            if score > threshold:
                G.add_edge(mid1, mid2)

        # Get refined components
        for component in nx.connected_components(G):
            refined_clusters.append(list(component))

    return refined_clusters


def tune_cross_encoder_threshold(cross_encoder, val_ids, mention_inputs,
                                 val_gold_clusters, bi_encoder_clusters,
                                 tokenizer, device, metric_fn=None):
    """
    Find optimal threshold for cross-encoder re-ranking on validation set.

    Args:
        cross_encoder: Trained CrossEncoderReranker
        val_ids: Validation mention IDs
        mention_inputs: Dict of mention texts
        val_gold_clusters: Ground truth validation clusters
        bi_encoder_clusters: Clusters from bi-encoder on validation set
        tokenizer: SciBERT tokenizer
        device: cuda/cpu
        metric_fn: Evaluation metric function

    Returns:
        best_threshold, best_f1
    """

    if metric_fn is None:
        metric_fn = evaluate_with_scorch

    best_threshold = 0.5
    best_f1 = 0

    print("Tuning cross-encoder threshold on validation set...")
    for threshold in np.arange(0.3, 0.8, 0.05):
        refined_clusters = rerank_clusters(
            bi_encoder_clusters, mention_inputs, cross_encoder,
            tokenizer, device, threshold=threshold
        )

        f1 = metric_fn(val_gold_clusters, refined_clusters)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    print(f"Optimal cross-encoder threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")
    return best_threshold, best_f1


## Hard Negative mining

In [ ]:

class HardNegativeDataset(Dataset):
    """
    Dataset with hard negative mining for improved contrastive learning.

    Hard negatives are pre-computed periodically (e.g., every epoch) to avoid
    excessive overhead while still benefiting from difficulty-based sampling.
    """
    def __init__(self, mention_ids, mention_inputs, mention_to_cluster,
                 model=None, tokenizer=None, device='cuda',
                 num_negatives=2, top_k_hard=10):
        """
        Args:
            mention_ids: List of mention IDs in this split
            mention_inputs: Dict mapping mention_id -> formatted text
            mention_to_cluster: Dict mapping mention_id -> cluster_id
            model: Pre-trained model for computing embeddings (optional)
            tokenizer: Tokenizer for encoding mentions
            device: Device for model inference
            num_negatives: Number of negatives to sample per anchor
            top_k_hard: Pool size of hard negatives to sample from
        """
        self.mention_ids = mention_ids
        self.mention_inputs = mention_inputs
        self.mention_to_cluster = mention_to_cluster
        self.num_negatives = num_negatives
        self.top_k_hard = top_k_hard
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        # Pre-compute embeddings for hard negative mining
        self.embeddings = None
        self.hard_negative_pool = None

        if model is not None and tokenizer is not None:
            self.update_hard_negatives(model, tokenizer, device)

    def update_hard_negatives(self, model, tokenizer, device):
        """
        Update hard negative pool by computing current embeddings.
        Call this periodically (e.g., every epoch) during training.
        """
        print("Computing embeddings for hard negative mining...")
        model.eval()
        embeddings = {}

        batch_size = 32
        with torch.no_grad():
            for i in tqdm(range(0, len(self.mention_ids), batch_size),
                         desc="Computing embeddings"):
                batch_ids = self.mention_ids[i:i+batch_size]
                batch_texts = [self.mention_inputs[mid] for mid in batch_ids]

                enc = tokenizer(
                    batch_texts, padding=True, truncation=True,
                    max_length=256, return_tensors='pt'
                ).to(device)

                with torch.amp.autocast('cuda'):
                    embs = model(enc['input_ids'], enc['attention_mask'])

                for mid, emb in zip(batch_ids, embs):
                    embeddings[mid] = emb.cpu().numpy()

        self.embeddings = embeddings

        # Pre-compute hard negative pool for each mention
        print("Building hard negative pools...")
        self.hard_negative_pool = {}

        for anchor_id in tqdm(self.mention_ids, desc="Finding hard negatives"):
            anchor_cluster = self.mention_to_cluster[anchor_id]
            anchor_emb = self.embeddings[anchor_id]

            # Compute similarities to all mentions from different clusters
            similarities = []
            for mid in self.mention_ids:
                if self.mention_to_cluster[mid] != anchor_cluster:
                    sim = np.dot(anchor_emb, self.embeddings[mid])
                    similarities.append((mid, sim))

            # Sort by similarity (descending) and take top-k
            similarities.sort(key=lambda x: x[1], reverse=True)
            hard_negs = [mid for mid, _ in similarities[:self.top_k_hard]]
            self.hard_negative_pool[anchor_id] = hard_negs

        print(f"Hard negative pools created. Avg pool size: {np.mean([len(pool) for pool in self.hard_negative_pool.values()]):.1f}")

    def __len__(self):
        return len(self.mention_ids)

    def __getitem__(self, idx):
        anchor_id = self.mention_ids[idx]
        anchor_cluster = self.mention_to_cluster[anchor_id]

        # Positive: random from same cluster
        same_cluster = [mid for mid in self.mention_ids
                       if self.mention_to_cluster[mid] == anchor_cluster
                       and mid != anchor_id]
        pos_id = random.choice(same_cluster) if same_cluster else anchor_id

        # Hard negatives: sample from pre-computed pool
        if self.hard_negative_pool is not None and anchor_id in self.hard_negative_pool:
            hard_pool = self.hard_negative_pool[anchor_id]
            if len(hard_pool) >= self.num_negatives:
                neg_ids = random.sample(hard_pool, self.num_negatives)
            else:
                # Fallback to random if not enough hard negatives
                diff_clusters = [mid for mid in self.mention_ids
                               if self.mention_to_cluster[mid] != anchor_cluster]
                neg_ids = random.sample(diff_clusters,
                                       min(self.num_negatives, len(diff_clusters)))
        else:
            # Fallback to random negatives
            diff_clusters = [mid for mid in self.mention_ids
                           if self.mention_to_cluster[mid] != anchor_cluster]
            neg_ids = random.sample(diff_clusters,
                                   min(self.num_negatives, len(diff_clusters)))

        return {
            'anchor_text': self.mention_inputs[anchor_id],
            'positive_text': self.mention_inputs[pos_id],
            'negative_texts': [self.mention_inputs[nid] for nid in neg_ids]
        }


def train_with_hard_negatives(model, train_dataset, val_ids, mention_inputs,
                              val_gold_clusters, tokenizer, device,
                              optimizer, scheduler, scaler, criterion,
                              num_epochs=50, patience=3, batch_size=8,
                              collate_fn=None, metric_fn=None):
    """
    Training loop with periodic hard negative updates.

    Args:
        model: SiameseSciBERT model
        train_dataset: HardNegativeDataset instance
        val_ids: Validation mention IDs
        mention_inputs: Dict of mention texts
        val_gold_clusters: Ground truth validation clusters
        tokenizer: SciBERT tokenizer
        device: cuda/cpu
        optimizer, scheduler, scaler, criterion: Training components
        num_epochs: Maximum epochs
        patience: Early stopping patience
        batch_size: Batch size
        collate_fn: Collate function
        metric_fn: Evaluation metric function (defaults to evaluate_with_scorch from train_refactored)

    Returns:
        best_val_f1, best_threshold
    """

    if metric_fn is None:
        metric_fn = evaluate_with_scorch

    best_val_f1 = 0
    patience_counter = 0
    best_threshold = 0.8

    # Update hard negatives before training starts
    train_dataset.update_hard_negatives(model, tokenizer, device)

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")

        # Update hard negatives every N epochs (e.g., every 3 epochs)
        if epoch > 1 and epoch % 3 == 0:
            print("Updating hard negative pools...")
            train_dataset.update_hard_negatives(model, tokenizer, device)

        # Create DataLoader (recreate to shuffle)
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=2,
            pin_memory=True
        )

        # Train epoch
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc="Training"):
            # Move to device
            anchor = {k: v.to(device) for k, v in batch['anchor'].items()}
            positive = {k: v.to(device) for k, v in batch['positive'].items()}
            negatives = {k: v.to(device) for k, v in batch['negatives'].items()}
            num_negatives = batch['num_negatives']

            optimizer.zero_grad(set_to_none=True)

            # Forward pass
            with torch.amp.autocast('cuda'):
                anchor_emb = model(anchor['input_ids'], anchor['attention_mask'])
                pos_emb = model(positive['input_ids'], positive['attention_mask'])
                neg_embs = model(negatives['input_ids'], negatives['attention_mask'])

                loss = criterion(anchor_emb, pos_emb, neg_embs, num_negatives)

            # Backward
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        if epoch == 1:

            best_threshold = find_optimal_threshold(
                model, val_ids, mention_inputs, val_gold_clusters,
                tokenizer, metric_fn
            )

        val_f1 = evaluate(
            model, val_ids, mention_inputs, val_gold_clusters,
            tokenizer, best_threshold, metric_fn
        )

        print(f"Epoch {epoch}: Train Loss = {avg_loss:.4f}, Val F1 = {val_f1:.4f}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), 'best_model_hard_neg.pth')
            print("✓ Best model saved.")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    return best_val_f1, best_threshold


## RUN COMPARISIONS

In [ ]:
import time


def variant_1_encoder():
    """
    Variant 1: Bi-Encoder
    - Cluster-based splitting
    - InfoNCE loss
    - Improved architecture
    """
    print("\n" + "="*80)
    print("VARIANT 1: Bi-Encoder")
    print("="*80)

    # Load data
    mentions, clusters, mention_to_cluster, cluster_to_mentions = load_data()
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}

    # Cluster-based split
    train_cluster_ids, val_cluster_ids, test_cluster_ids = cluster_based_split(clusters)
    train_ids = [mid for cid in train_cluster_ids for mid in clusters[cid]]
    val_ids = [mid for cid in val_cluster_ids for mid in clusters[cid]]
    test_ids = [mid for cid in test_cluster_ids for mid in clusters[cid]]

    val_gold_clusters = [clusters[cid] for cid in val_cluster_ids]
    test_gold_clusters = [clusters[cid] for cid in test_cluster_ids]

    # Initialize
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.amp.GradScaler('cuda')
    criterion = InfoNCELoss(temperature=0.07)

    # Dataset
    train_dataset = CorefPairDataset(train_ids, mention_inputs, mention_to_cluster)

    def collate_with_tokenizer(batch):
        return collate_fn(batch, tokenizer)

    from torch.utils.data import DataLoader
    train_loader = DataLoader(
        train_dataset, batch_size=8, shuffle=True,
        collate_fn=collate_with_tokenizer, num_workers=2, pin_memory=True
    )

    # Training
    num_epochs = 50
    patience = 3
    best_val_f1 = 0
    patience_counter = 0
    best_threshold = 0.8

    epoch_times = []
    start_total = time.time()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        # Evaluate
        if epoch == 1:
            best_threshold = find_optimal_threshold(
                model, val_ids, mention_inputs, val_gold_clusters,
                tokenizer, evaluate_with_scorch
            )

        val_f1 = evaluate(
            model, val_ids, mention_inputs, val_gold_clusters,
            tokenizer, best_threshold, evaluate_with_scorch
        )

        print(f"Epoch {epoch}: Loss={train_loss:.4f}, Val F1={val_f1:.4f}, Time={epoch_time:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), 'best_model_variant1.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    total_train_time = time.time() - start_total

    # Test evaluation
    model.load_state_dict(torch.load('best_model_variant1.pth'))
    best_threshold = find_optimal_threshold(
        model, val_ids, mention_inputs, val_gold_clusters,
        tokenizer, evaluate_with_scorch
    )

    inference_start = time.time()
    test_f1 = evaluate(
        model, test_ids, mention_inputs, test_gold_clusters,
        tokenizer, best_threshold, evaluate_with_scorch
    )
    inference_time = time.time() - inference_start

    return {
        'variant': 'Baseline',
        'conll_f1': test_f1,
        'notes': 'Improved dataset, loss, training loop, architecture'
    }


def variant_2_hard_negatives():
    """
    Variant 2: + Hard Negative Mining
    """
    print("\n" + "="*80)
    print("VARIANT 2: Baseline + Hard Negative Mining")
    print("="*80)

    # Load data (same as variant 1)
    mentions, clusters, mention_to_cluster, cluster_to_mentions = load_data()
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}

    train_cluster_ids, val_cluster_ids, test_cluster_ids = cluster_based_split(clusters)
    train_ids = [mid for cid in train_cluster_ids for mid in clusters[cid]]
    val_ids = [mid for cid in val_cluster_ids for mid in clusters[cid]]
    test_ids = [mid for cid in test_cluster_ids for mid in clusters[cid]]

    val_gold_clusters = [clusters[cid] for cid in val_cluster_ids]
    test_gold_clusters = [clusters[cid] for cid in test_cluster_ids]

    # Initialize
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.amp.GradScaler('cuda')
    criterion = InfoNCELoss(temperature=0.07)

    # Hard negative dataset
    train_dataset = HardNegativeDataset(
        train_ids, mention_inputs, mention_to_cluster,
        model=model, tokenizer=tokenizer, device=device,
        num_negatives=2, top_k_hard=10
    )

    def collate_with_tokenizer(batch):
        return collate_fn(batch, tokenizer)

    # Training with hard negatives
    start_total = time.time()

    best_val_f1, best_threshold = train_with_hard_negatives(
        model, train_dataset, val_ids, mention_inputs, val_gold_clusters,
        tokenizer, device, optimizer, scheduler, scaler, criterion,
        num_epochs=50, patience=3, batch_size=8,
        collate_fn=collate_with_tokenizer, metric_fn=evaluate_with_scorch
    )

    total_train_time = time.time() - start_total

    # Test evaluation
    model.load_state_dict(torch.load('best_model_hard_neg.pth'))

    inference_start = time.time()
    test_f1 = evaluate(
        model, test_ids, mention_inputs, test_gold_clusters,
        tokenizer, best_threshold, evaluate_with_scorch
    )
    inference_time = time.time() - inference_start

    return {
        'variant': '+Hard Negatives',
        'conll_f1': test_f1,
        'notes': 'Periodic hard negative mining (every 3 epochs)'
    }


def variant_3_cross_encoder():
    """
    Variant 3: + Cross-Encoder Re-Ranking
    """
    print("\n" + "="*80)
    print("VARIANT 3: + Cross-Encoder Re-Ranking")
    print("="*80)

    # Load bi-encoder from variant 2
    mentions, clusters, mention_to_cluster, cluster_to_mentions = load_data()
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}

    train_cluster_ids, val_cluster_ids, test_cluster_ids = cluster_based_split(clusters)
    train_ids = [mid for cid in train_cluster_ids for mid in clusters[cid]]
    val_ids = [mid for cid in val_cluster_ids for mid in clusters[cid]]
    test_ids = [mid for cid in test_cluster_ids for mid in clusters[cid]]

    val_gold_clusters = [clusters[cid] for cid in val_cluster_ids]
    test_gold_clusters = [clusters[cid] for cid in test_cluster_ids]

    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Load bi-encoder
    bi_encoder = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    bi_encoder.load_state_dict(torch.load('best_model_variant1.pth'))

    # Train cross-encoder
    print("\nTraining cross-encoder...")

    train_ce_dataset = PairwiseCorefDataset(
        train_ids, mention_inputs, mention_to_cluster,
        {cid: clusters[cid] for cid in train_cluster_ids}
    )

    val_ce_dataset = PairwiseCorefDataset(
        val_ids, mention_inputs, mention_to_cluster,
        {cid: clusters[cid] for cid in val_cluster_ids}
    )

    start_ce_train = time.time()
    cross_encoder = train_cross_encoder(
        train_ce_dataset, val_ce_dataset, tokenizer, device,
        num_epochs=15, batch_size=8, lr=2e-5
    )
    ce_train_time = time.time() - start_ce_train

    # Get bi-encoder clusters on validation set
    print("\nGetting bi-encoder clusters on validation...")
    val_embeddings = get_embeddings(bi_encoder, val_ids, mention_inputs, tokenizer)
    val_bi_clusters = cluster_mentions(val_embeddings, threshold=0.8)

    # Tune cross-encoder threshold
    ce_threshold, _ = tune_cross_encoder_threshold(
        cross_encoder, val_ids, mention_inputs, val_gold_clusters,
        val_bi_clusters, tokenizer, device, evaluate_with_scorch
    )

    # Test evaluation
    print("\nEvaluating on test set...")
    test_embeddings = get_embeddings(bi_encoder, test_ids, mention_inputs, tokenizer)
    test_bi_clusters = cluster_mentions(test_embeddings, threshold=0.8)

    inference_start = time.time()
    test_refined_clusters = rerank_clusters(
        test_bi_clusters, mention_inputs, cross_encoder,
        tokenizer, device, threshold=ce_threshold
    )
    inference_time = time.time() - inference_start

    test_f1 = evaluate_with_scorch(test_gold_clusters, test_refined_clusters)

    return {
        'variant': '+Cross-Encoder',
        'conll_f1': test_f1,
        'notes': 'Bi-encoder + cross-encoder re-ranking'
    }


def main():
    """Run all variants and save comparison results"""
    print("="*80)
    print("COMPARISON: SciBERT Coreference Resolution")
    print("="*80)

    results = []

    # Run each variant
    try:
        results.append(variant_1_encoder())
    except Exception as e:
        print(f"Error in variant 2: {e}")

    try:
        results.append(variant_2_hard_negatives())
    except Exception as e:
        print(f"Error in variant 2: {e}")

    try:
        results.append(variant_3_cross_encoder())
    except Exception as e:
        print(f"Error in variant 3: {e}")

    # Create results dataframe
    df = pd.DataFrame(results)

    # Save to CSV
    df.to_csv('comparison_results.csv', index=False)
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(df.to_string(index=False))
    print("\nResults saved to comparison_results.csv")

    # Save detailed JSON
    with open('comparison_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print("Detailed results saved to comparison_results.json")


if __name__ == "__main__":
    main()



COMPARISON: SciBERT Coreference Resolution

VARIANT 1: Bi-Encoder
Loading data...
Loaded 2974 mentions and 733 clusters.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 69157.88it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Getting embeddings: 100%|██████████| 12/12 [00:00<00:00, 154.85it/

Tuning threshold on validation set...
Optimal threshold: 0.70, F1: 0.9507


Getting embeddings: 100%|██████████| 12/12 [00:00<00:00, 162.18it/s]


Epoch 1: Loss=0.0899, Val F1=0.9507, Time=11.0s


...

Tuning cross-encoder threshold on validation set...


Re-ranking clusters: 100%|██████████| 80/80 [00:00<00:00, 327.58it/s]


Optimal cross-encoder threshold: 0.30, F1: 0.9586

Evaluating on test set...


Re-ranking clusters: 100%|██████████| 78/78 [00:23<00:00,  3.30it/s]



FINAL RESULTS
        variant  conll_f1                                               notes
       Baseline  0.986015 Improved dataset, loss, training loop, architecture
+Hard Negatives  0.987871      Periodic hard negative mining (every 3 epochs)
 +Cross-Encoder  0.986015               Bi-encoder + cross-encoder re-ranking

Results saved to comparison_results.csv
Detailed results saved to comparison_results.json


## Predicting the tests for the Baseline model

In [ ]:
# from cross_encoder import CrossEncoderReranker, rerank_clusters

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CONFIGURATION - Edit these values to customize behavior
# ============================================================================
CONFIG = {
    'model_path': 'best_model_variant1.pth',           # Path to trained bi-encoder
    'test_data_path': 'test_data.jsonl', # Path to test data
    'threshold': 0.8,                                     # Clustering threshold (None = auto 0.8)
    'cross_encoder_path': None,                          # Path to cross-encoder (None = disabled)
    'ce_threshold': 0.5,                                 # Cross-encoder threshold
    'output_path': 'clusters_with_variant1.json',                      # Output clusters file
}
# ============================================================================


def load_test_data(test_data_path='test_data.jsonl'):
    """
    Load test data (without labels).

    Returns:
        mentions: Dict mapping mention_id -> mention data
        mention_inputs: Dict mapping mention_id -> preprocessed text
        mention_ids: List of all mention IDs
    """
    print(f"Loading test data from {test_data_path}...")

    mentions = {}
    with open(test_data_path, 'r') as f:
        for line in f:
            mention_data = json.loads(line.strip())
            mentions[mention_data['mention_id']] = mention_data

    # Preprocess mentions
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}
    mention_ids = list(mentions.keys())

    print(f"Loaded {len(mentions)} test mentions.")
    return mentions, mention_inputs, mention_ids


def predict_clusters(model_path, test_data_path='test_data.jsonl',
                     threshold=None, cross_encoder_path=None, ce_threshold=0.5):
    """
    Predict clusters for test data using trained model.

    Args:
        model_path: Path to trained bi-encoder model (.pth file)
        test_data_path: Path to test_data.jsonl
        threshold: Clustering threshold (if None, uses 0.8 as default)
        cross_encoder_path: Optional path to cross-encoder for re-ranking
        ce_threshold: Cross-encoder threshold (default 0.5)

    Returns:
        predicted_clusters: List of lists, each containing mention IDs in a cluster
    """
    # Load test data
    mentions, mention_inputs, mention_ids = load_test_data(test_data_path)

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Load bi-encoder model
    print(f"Loading bi-encoder model from {model_path}...")
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Get embeddings
    print("Generating embeddings...")
    embeddings = get_embeddings(model, mention_ids, mention_inputs, tokenizer, batch_size=16)

    # Cluster using bi-encoder
    if threshold is None:
        threshold = 0.8  # Default threshold
        print(f"Using default threshold: {threshold}")
    else:
        print(f"Using threshold: {threshold}")

    print("Clustering mentions...")
    predicted_clusters = cluster_mentions(embeddings, threshold=threshold)
    print(f"Found {len(predicted_clusters)} clusters")

    # Optional: Re-rank with cross-encoder
    if cross_encoder_path is not None:
        print(f"\nLoading cross-encoder from {cross_encoder_path}...")
        cross_encoder = CrossEncoderReranker().to(device)
        cross_encoder.load_state_dict(torch.load(cross_encoder_path, map_location=device))
        cross_encoder.eval()

        print(f"Re-ranking clusters with cross-encoder (threshold={ce_threshold})...")
        predicted_clusters = rerank_clusters(
            predicted_clusters, mention_inputs, cross_encoder,
            tokenizer, device, threshold=ce_threshold
        )
        print(f"After re-ranking: {len(predicted_clusters)} clusters")

    return predicted_clusters


def save_clusters(predicted_clusters, output_path='clusters.json'):
    """
    Save predicted clusters to JSON file.

    Format: List of lists, where each sublist contains mention IDs in a cluster.
    Example: [["m1", "m2", "m3"], ["m4", "m5"], ["m6"]]

    Args:
        predicted_clusters: List of lists of mention IDs
        output_path: Path to output JSON file
    """
    print(f"\nSaving clusters to {output_path}...")

    with open(output_path, 'w') as f:
        json.dump(predicted_clusters, f, indent=2)

    # Print statistics
    cluster_sizes = [len(cluster) for cluster in predicted_clusters]
    print(f"\nCluster Statistics:")
    print(f"  Total clusters: {len(predicted_clusters)}")
    print(f"  Singleton clusters: {sum(1 for size in cluster_sizes if size == 1)}")
    print(f"  Average cluster size: {sum(cluster_sizes) / len(cluster_sizes):.2f}")
    print(f"  Largest cluster: {max(cluster_sizes)}")
    print(f"\nClusters saved successfully!")


"""Run inference with configuration from CONFIG dictionary."""
print("=" * 80)
print("SciBERT Coreference Resolution - Test Inference")
print("=" * 80)
print(f"Model: {CONFIG['model_path']}")
print(f"Test data: {CONFIG['test_data_path']}")
print(f"Threshold: {CONFIG['threshold']}")
print(f"Output: {CONFIG['output_path']}")
if CONFIG['cross_encoder_path']:
    print(f"Cross-encoder: {CONFIG['cross_encoder_path']}")
    print(f"Cross-encoder threshold: {CONFIG['ce_threshold']}")
print("=" * 80)

# Predict clusters
predicted_clusters = predict_clusters(
    model_path=CONFIG['model_path'],
    test_data_path=CONFIG['test_data_path'],
    threshold=CONFIG['threshold'],
    cross_encoder_path=CONFIG['cross_encoder_path'],
    ce_threshold=CONFIG['ce_threshold']
)

# Save to JSON
save_clusters(predicted_clusters, output_path=CONFIG['output_path'])

print("\n" + "=" * 80)
print("Inference complete!")
print("=" * 80)




SciBERT Coreference Resolution - Test Inference
Model: best_model_variant1.pth
Test data: test_data.jsonl
Threshold: 0.8
Output: clusters_with_variant1.json
Loading test data from test_data.jsonl...
Loaded 743 test mentions.
Loading tokenizer...
Loading bi-encoder model from best_model_variant1.pth...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 59961.67it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Getting embeddings: 100%|██████████| 47/47 [00:00<00:00, 168.61it/s]


Using threshold: 0.8
Clustering mentions...
Found 245 clusters

Saving clusters to clusters_with_variant1.json...

Cluster Statistics:
  Total clusters: 245
  Singleton clusters: 137
  Average cluster size: 3.03
  Largest cluster: 73

Clusters saved successfully!

Inference complete!


## Predicting the tests for hard negative mining variant

In [ ]:
# from cross_encoder import CrossEncoderReranker, rerank_clusters

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CONFIGURATION - Edit these values to customize behavior
# ============================================================================
CONFIG = {
    'model_path': 'best_model_hard_neg.pth',           # Path to trained bi-encoder
    'test_data_path': 'test_data.jsonl', # Path to test data
    'threshold': 0.8,                                     # Clustering threshold (None = auto 0.8)
    'cross_encoder_path': None,                          # Path to cross-encoder (None = disabled)
    'ce_threshold': 0.5,                                 # Cross-encoder threshold
    'output_path': 'clusters_with_hard_negatvie_mining.json',                      # Output clusters file
}
# ============================================================================


def load_test_data(test_data_path='test_data.jsonl'):
    """
    Load test data (without labels).

    Returns:
        mentions: Dict mapping mention_id -> mention data
        mention_inputs: Dict mapping mention_id -> preprocessed text
        mention_ids: List of all mention IDs
    """
    print(f"Loading test data from {test_data_path}...")

    mentions = {}
    with open(test_data_path, 'r') as f:
        for line in f:
            mention_data = json.loads(line.strip())
            mentions[mention_data['mention_id']] = mention_data

    # Preprocess mentions
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}
    mention_ids = list(mentions.keys())

    print(f"Loaded {len(mentions)} test mentions.")
    return mentions, mention_inputs, mention_ids


def predict_clusters(model_path, test_data_path='test_data.jsonl',
                     threshold=None, cross_encoder_path=None, ce_threshold=0.5):
    """
    Predict clusters for test data using trained model.

    Args:
        model_path: Path to trained bi-encoder model (.pth file)
        test_data_path: Path to test_data.jsonl
        threshold: Clustering threshold (if None, uses 0.8 as default)
        cross_encoder_path: Optional path to cross-encoder for re-ranking
        ce_threshold: Cross-encoder threshold (default 0.5)

    Returns:
        predicted_clusters: List of lists, each containing mention IDs in a cluster
    """
    # Load test data
    mentions, mention_inputs, mention_ids = load_test_data(test_data_path)

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Load bi-encoder model
    print(f"Loading bi-encoder model from {model_path}...")
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Get embeddings
    print("Generating embeddings...")
    embeddings = get_embeddings(model, mention_ids, mention_inputs, tokenizer, batch_size=16)

    # Cluster using bi-encoder
    if threshold is None:
        threshold = 0.8  # Default threshold
        print(f"Using default threshold: {threshold}")
    else:
        print(f"Using threshold: {threshold}")

    print("Clustering mentions...")
    predicted_clusters = cluster_mentions(embeddings, threshold=threshold)
    print(f"Found {len(predicted_clusters)} clusters")

    # Optional: Re-rank with cross-encoder
    if cross_encoder_path is not None:
        print(f"\nLoading cross-encoder from {cross_encoder_path}...")
        cross_encoder = CrossEncoderReranker().to(device)
        cross_encoder.load_state_dict(torch.load(cross_encoder_path, map_location=device))
        cross_encoder.eval()

        print(f"Re-ranking clusters with cross-encoder (threshold={ce_threshold})...")
        predicted_clusters = rerank_clusters(
            predicted_clusters, mention_inputs, cross_encoder,
            tokenizer, device, threshold=ce_threshold
        )
        print(f"After re-ranking: {len(predicted_clusters)} clusters")

    return predicted_clusters


def save_clusters(predicted_clusters, output_path='clusters.json'):
    """
    Save predicted clusters to JSON file.

    Format: List of lists, where each sublist contains mention IDs in a cluster.
    Example: [["m1", "m2", "m3"], ["m4", "m5"], ["m6"]]

    Args:
        predicted_clusters: List of lists of mention IDs
        output_path: Path to output JSON file
    """
    print(f"\nSaving clusters to {output_path}...")

    with open(output_path, 'w') as f:
        json.dump(predicted_clusters, f, indent=2)

    # Print statistics
    cluster_sizes = [len(cluster) for cluster in predicted_clusters]
    print(f"\nCluster Statistics:")
    print(f"  Total clusters: {len(predicted_clusters)}")
    print(f"  Singleton clusters: {sum(1 for size in cluster_sizes if size == 1)}")
    print(f"  Average cluster size: {sum(cluster_sizes) / len(cluster_sizes):.2f}")
    print(f"  Largest cluster: {max(cluster_sizes)}")
    print(f"\nClusters saved successfully!")


"""Run inference with configuration from CONFIG dictionary."""
print("=" * 80)
print("SciBERT Coreference Resolution - Test Inference")
print("=" * 80)
print(f"Model: {CONFIG['model_path']}")
print(f"Test data: {CONFIG['test_data_path']}")
print(f"Threshold: {CONFIG['threshold']}")
print(f"Output: {CONFIG['output_path']}")
if CONFIG['cross_encoder_path']:
    print(f"Cross-encoder: {CONFIG['cross_encoder_path']}")
    print(f"Cross-encoder threshold: {CONFIG['ce_threshold']}")
print("=" * 80)

# Predict clusters
predicted_clusters = predict_clusters(
    model_path=CONFIG['model_path'],
    test_data_path=CONFIG['test_data_path'],
    threshold=CONFIG['threshold'],
    cross_encoder_path=CONFIG['cross_encoder_path'],
    ce_threshold=CONFIG['ce_threshold']
)

# Save to JSON
save_clusters(predicted_clusters, output_path=CONFIG['output_path'])

print("\n" + "=" * 80)
print("Inference complete!")
print("=" * 80)




SciBERT Coreference Resolution - Test Inference
Model: best_model_hard_neg.pth
Test data: test_data.jsonl
Threshold: 0.8
Output: clusters_with_hard_negatvie_mining.json
Loading test data from test_data.jsonl...
Loaded 743 test mentions.
Loading tokenizer...
Loading bi-encoder model from best_model_hard_neg.pth...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 59474.60it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Getting embeddings: 100%|██████████| 47/47 [00:00<00:00, 165.77it/s]


Using threshold: 0.8
Clustering mentions...
Found 251 clusters

Saving clusters to clusters_with_hard_negatvie_mining.json...

Cluster Statistics:
  Total clusters: 251
  Singleton clusters: 142
  Average cluster size: 2.96
  Largest cluster: 73

Clusters saved successfully!

Inference complete!


## Predicting the tests for cross encoder variant

In [ ]:
# from cross_encoder import CrossEncoderReranker, rerank_clusters

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CONFIGURATION - Edit these values to customize behavior
# ============================================================================
CONFIG = {
    'model_path': 'best_model_variant1.pth',           # Path to trained bi-encoder
    'test_data_path': 'test_data.jsonl', # Path to test data
    'threshold': 0.8,                                     # Clustering threshold (None = auto 0.8)
    'cross_encoder_path': 'best_cross_encoder.pth',                          # Path to cross-encoder (None = disabled)
    'ce_threshold': 0.5,                                 # Cross-encoder threshold
    'output_path': 'clusters_with_cross_encoder.json',                      # Output clusters file
}
# ============================================================================


def load_test_data(test_data_path='test_data.jsonl'):
    """
    Load test data (without labels).

    Returns:
        mentions: Dict mapping mention_id -> mention data
        mention_inputs: Dict mapping mention_id -> preprocessed text
        mention_ids: List of all mention IDs
    """
    print(f"Loading test data from {test_data_path}...")

    mentions = {}
    with open(test_data_path, 'r') as f:
        for line in f:
            mention_data = json.loads(line.strip())
            mentions[mention_data['mention_id']] = mention_data

    # Preprocess mentions
    mention_inputs = {mid: preprocess_mention(data) for mid, data in mentions.items()}
    mention_ids = list(mentions.keys())

    print(f"Loaded {len(mentions)} test mentions.")
    return mentions, mention_inputs, mention_ids


def predict_clusters(model_path, test_data_path='test_data.jsonl',
                     threshold=None, cross_encoder_path=None, ce_threshold=0.5):
    """
    Predict clusters for test data using trained model.

    Args:
        model_path: Path to trained bi-encoder model (.pth file)
        test_data_path: Path to test_data.jsonl
        threshold: Clustering threshold (if None, uses 0.8 as default)
        cross_encoder_path: Optional path to cross-encoder for re-ranking
        ce_threshold: Cross-encoder threshold (default 0.5)

    Returns:
        predicted_clusters: List of lists, each containing mention IDs in a cluster
    """
    # Load test data
    mentions, mention_inputs, mention_ids = load_test_data(test_data_path)

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

    # Load bi-encoder model
    print(f"Loading bi-encoder model from {model_path}...")
    model = SiameseSciBERT(projection_dim=256, dropout=0.1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Get embeddings
    print("Generating embeddings...")
    embeddings = get_embeddings(model, mention_ids, mention_inputs, tokenizer, batch_size=16)

    # Cluster using bi-encoder
    if threshold is None:
        threshold = 0.8  # Default threshold
        print(f"Using default threshold: {threshold}")
    else:
        print(f"Using threshold: {threshold}")

    print("Clustering mentions...")
    predicted_clusters = cluster_mentions(embeddings, threshold=threshold)
    print(f"Found {len(predicted_clusters)} clusters")

    # Optional: Re-rank with cross-encoder
    if cross_encoder_path is not None:
        print(f"\nLoading cross-encoder from {cross_encoder_path}...")
        cross_encoder = CrossEncoderReranker().to(device)
        cross_encoder.load_state_dict(torch.load(cross_encoder_path, map_location=device))
        cross_encoder.eval()

        print(f"Re-ranking clusters with cross-encoder (threshold={ce_threshold})...")
        predicted_clusters = rerank_clusters(
            predicted_clusters, mention_inputs, cross_encoder,
            tokenizer, device, threshold=ce_threshold
        )
        print(f"After re-ranking: {len(predicted_clusters)} clusters")

    return predicted_clusters


def save_clusters(predicted_clusters, output_path='clusters.json'):
    """
    Save predicted clusters to JSON file.

    Format: List of lists, where each sublist contains mention IDs in a cluster.
    Example: [["m1", "m2", "m3"], ["m4", "m5"], ["m6"]]

    Args:
        predicted_clusters: List of lists of mention IDs
        output_path: Path to output JSON file
    """
    print(f"\nSaving clusters to {output_path}...")

    with open(output_path, 'w') as f:
        json.dump(predicted_clusters, f, indent=2)

    # Print statistics
    cluster_sizes = [len(cluster) for cluster in predicted_clusters]
    print(f"\nCluster Statistics:")
    print(f"  Total clusters: {len(predicted_clusters)}")
    print(f"  Singleton clusters: {sum(1 for size in cluster_sizes if size == 1)}")
    print(f"  Average cluster size: {sum(cluster_sizes) / len(cluster_sizes):.2f}")
    print(f"  Largest cluster: {max(cluster_sizes)}")
    print(f"\nClusters saved successfully!")


"""Run inference with configuration from CONFIG dictionary."""
print("=" * 80)
print("SciBERT Coreference Resolution - Test Inference")
print("=" * 80)
print(f"Model: {CONFIG['model_path']}")
print(f"Test data: {CONFIG['test_data_path']}")
print(f"Threshold: {CONFIG['threshold']}")
print(f"Output: {CONFIG['output_path']}")
if CONFIG['cross_encoder_path']:
    print(f"Cross-encoder: {CONFIG['cross_encoder_path']}")
    print(f"Cross-encoder threshold: {CONFIG['ce_threshold']}")
print("=" * 80)

# Predict clusters
predicted_clusters = predict_clusters(
    model_path=CONFIG['model_path'],
    test_data_path=CONFIG['test_data_path'],
    threshold=CONFIG['threshold'],
    cross_encoder_path=CONFIG['cross_encoder_path'],
    ce_threshold=CONFIG['ce_threshold']
)

# Save to JSON
save_clusters(predicted_clusters, output_path=CONFIG['output_path'])

print("\n" + "=" * 80)
print("Inference complete!")
print("=" * 80)




SciBERT Coreference Resolution - Test Inference
Model: best_model_variant1.pth
Test data: test_data.jsonl
Threshold: 0.8
Output: clusters_with_cross_encoder.json
Cross-encoder: best_cross_encoder.pth
Cross-encoder threshold: 0.5
Loading test data from test_data.jsonl...
Loaded 743 test mentions.
Loading tokenizer...
Loading bi-encoder model from best_model_variant1.pth...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 68303.31it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Getting embeddings: 100%|██████████| 47/47 [00:00<00:00, 162.26it/s]


Using threshold: 0.8
Clustering mentions...
Found 243 clusters

Loading cross-encoder from best_cross_encoder.pth...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 71393.94it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Re-ranking clusters with cross-encoder (threshold=0.5)...


Re-ranking clusters: 100%|██████████| 243/243 [00:02<00:00, 87.66it/s] 

After re-ranking: 248 clusters

Saving clusters to clusters_with_cross_encoder.json...

Cluster Statistics:
  Total clusters: 248
  Singleton clusters: 137
  Average cluster size: 3.00
  Largest cluster: 73

Clusters saved successfully!

Inference complete!
